# Notebook 4 — Ensemble & Hybrid Model

Combines Isolation Forest + LSTM via soft-voting ensemble. Reproduces all paper figures and Table 2 comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_generator import generate_dataset
from src.preprocessor   import PipelinePreprocessor
from src.models         import (
    IsolationForestDetector, LSTMAutoencoderDetector,
    EnsembleDetector, compare_models
)
from src.visualizer import (
    plot_fig2_swarm, plot_fig3_histogram,
    plot_anomaly_scores, plot_model_comparison
)

In [ ]:
df   = generate_dataset(seed=42)
prep = PipelinePreprocessor(window_size=10)
X    = prep.fit_transform(df)
X_seq = prep.make_sequences(X)
y    = df['is_anomaly'].values
y_seq = y[9:]

In [ ]:
# Train models
ifd  = IsolationForestDetector(contamination=0.054, n_estimators=200)
lstm = LSTMAutoencoderDetector(window_size=10)
ifd.fit(X[y == 0])
lstm.fit(X_seq[y_seq == 0], epochs=20)

ensemble = EnsembleDetector(ifd, lstm, window_size=10)

In [ ]:
# Evaluate all models and print comparison (Table 2)
all_results = ensemble.evaluate(X, X_seq, y)
compare_models(all_results)

In [ ]:
# Add predictions to dataframe
ensemble_scores = ensemble.score_samples(X, X_seq)
df['ensemble_score']   = ensemble_scores
df['ensemble_anomaly'] = (ensemble_scores > 0.5).astype(int)

# Paper figures — saved to results/
plot_fig2_swarm(df)
plot_fig3_histogram(df)
plot_anomaly_scores(df, ensemble_scores)
plot_model_comparison(all_results)
print('All figures saved to results/')

In [ ]:
# Display the timeline figure inline
from IPython.display import Image
Image('../results/anomaly_scores_timeline.png')